# 🎥 Video Understanding Pipeline — Kaggle (v3.1 Final)

**Models**: Moondream2, SmolVLM, Qwen2.5-VL-3B, Qwen3-VL-2B
**Features**: Multi-Model, Dual GPU Parallelism, Real-Time Streaming, **Optimized Inference**

### ⚠️ Before running:
1. **Accelerator → GPU T4 x2**
2. **Settings → Internet → ON**
3. Run all cells in order

In [ ]:
# ── 1. Install ───────────────────────────────────────────────
!apt-get update -qq && apt-get install -y -qq libvips-dev > /dev/null 2>&1
!pip install -q --upgrade transformers accelerate
!pip install -q fastapi uvicorn python-multipart opencv-python-headless \
    Pillow einops pyngrok pyvips qwen_vl_utils
print('✅ All packages installed')

In [ ]:
# ── 2. Verify GPUs & Optimization ────────────────────────────
import torch
if not torch.cuda.is_available():
    raise RuntimeError('❌ GPU not enabled! Sidebar → Accelerator → GPU T4 x2')

# 🚀 Global Optimizations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

n_gpus = torch.cuda.device_count()
for i in range(n_gpus):
    n = torch.cuda.get_device_name(i)
    v = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {n} ({v:.1f} GB)')
if n_gpus < 2:
    print('⚠️ Only 1 GPU found. Dual GPU mode disabled.')
else:
    print(f'✅ Dual GPU mode active ({n_gpus} GPUs) + Optimizations Enabled')

In [ ]:
# ── 3. Model Manager (Dual GPU + Optimized) ──────────────────
import gc, time, os, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image

# Monkey-patch for Moondream2 compatibility
from transformers import modeling_utils as _tmu
if not hasattr(_tmu.PreTrainedModel, '_patched_mark'):
    _orig_mark = _tmu.PreTrainedModel.mark_tied_weights_as_initialized
    def _patched_mark(self):
        if not hasattr(self, 'all_tied_weights_keys'):
            self.all_tied_weights_keys = {}
        return _orig_mark(self)
    _tmu.PreTrainedModel.mark_tied_weights_as_initialized = _patched_mark
    _tmu.PreTrainedModel._patched_mark = True

MODEL_REGISTRY = {
    "moondream2": {"hf_id": "vikhyatk/moondream2", "label": "Moondream2 (~4GB)"},
    "smolvlm":   {"hf_id": "HuggingFaceTB/SmolVLM-Instruct", "label": "SmolVLM (~4GB)"},
    "qwen25vl":  {"hf_id": "Qwen/Qwen2.5-VL-3B-Instruct", "label": "Qwen2.5-VL 3B (~6GB)"},
    "qwen3vl":   {"hf_id": "Qwen/Qwen3-VL-2B-Instruct", "label": "Qwen3-VL 2B (~4GB)"},
}

class ModelManager:
    def __init__(self):
        self.current = None
        self.models = []  # List of models (one per GPU)
        self.processors = []
        self.lock = threading.Lock()

    def unload(self):
        with self.lock:
            if self.models:
                del self.models, self.processors
                self.models, self.processors = [], []
                self.current = None
                gc.collect()
                torch.cuda.empty_cache()
                print('  🗑️ Previous models unloaded')

    def load(self, name):
        if name == self.current:
            return
        info = MODEL_REGISTRY[name]
        print(f'⏳ Loading {info["label"]} on both GPUs...')
        self.unload()
        t0 = time.time()
        
        n_gpus = torch.cuda.device_count()
        if n_gpus == 0: n_gpus = 1  # CPU fallback
        
        for i in range(n_gpus):
            device = f"cuda:{i}" if torch.cuda.is_available() else "cpu"
            print(f"  Loading on {device}...")
            
            if name == "moondream2":
                from transformers import AutoModelForCausalLM
                # ⚠️ IMPORTANT: No device_map here to avoid caching_allocator_warmup bug
                m = AutoModelForCausalLM.from_pretrained(
                    info["hf_id"], revision="2025-01-09", trust_remote_code=True,
                    torch_dtype=torch.float16
                ).to(device).eval()
                self.models.append(m)
                self.processors.append(None)
            
            elif name == "smolvlm":
                from transformers import AutoProcessor, AutoModelForVision2Seq
                p = AutoProcessor.from_pretrained(info["hf_id"])
                m = AutoModelForVision2Seq.from_pretrained(
                    info["hf_id"], torch_dtype=torch.float16,
                    _attn_implementation="eager"
                ).to(device).eval()
                self.models.append(m)
                self.processors.append(p)
            
            elif name in ("qwen25vl", "qwen3vl"):
                if name == "qwen25vl":
                    from transformers import Qwen2_5_VLForConditionalGeneration as cls
                else:
                    from transformers import Qwen3VLForConditionalGeneration as cls
                from transformers import AutoProcessor
                p = AutoProcessor.from_pretrained(info["hf_id"])
                m = cls.from_pretrained(
                    info["hf_id"], torch_dtype=torch.float16, device_map={"" : device}).eval()
                self.models.append(m)
                self.processors.append(p)

        self.current = name
        print(f'✅ Loaded on {len(self.models)} GPU(s) in {time.time()-t0:.1f}s')

    @torch.inference_mode()  # 🚀 Optimization: No grad = faster + less RAM
    def query_single(self, image, prompt, gpu_idx):
        """Run inference on specific GPU."""
        gpu_idx = gpu_idx % len(self.models)
        device = self.models[gpu_idx].device
        model = self.models[gpu_idx]
        processor = self.processors[gpu_idx]
        
        try:
            if self.current == "moondream2":
                ans = model.query(image, prompt)
                if isinstance(ans, dict) and 'answer' in ans: return ans['answer'].strip()
                return str(ans).strip()
            
            elif self.current == "smolvlm":
                msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
                text = processor.apply_chat_template(msgs, add_generation_prompt=True)
                inputs = processor(text=text, images=[image], return_tensors="pt").to(device)
                # 🚀 Optimization: use_cache=True
                ids = model.generate(**inputs, max_new_tokens=256, use_cache=True)
                out = processor.batch_decode(ids, skip_special_tokens=True)[0]
                if "assistant" in out.lower(): out = out.split("assistant")[-1]
                return out.strip()
            
            elif self.current in ("qwen25vl", "qwen3vl"):
                import tempfile
                with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
                    image.save(tmp.name)
                    tmp_name = tmp.name
                try:
                    msgs = [{"role": "user", "content": [
                        {"type": "image", "image": f"file://{tmp_name}"},
                        {"type": "text", "text": prompt}
                    ]}]
                    if self.current == "qwen25vl":
                        from qwen_vl_utils import process_vision_info
                        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
                        img_in, vid_in = process_vision_info(msgs)
                        inputs = processor(text=[text], images=img_in, videos=vid_in,
                                          padding=True, return_tensors="pt").to(device)
                    else:
                        inputs = processor.apply_chat_template(
                            msgs, tokenize=True, add_generation_prompt=True,
                            return_dict=True, return_tensors="pt").to(device)
                    
                    ids = model.generate(**inputs, max_new_tokens=256, use_cache=True)
                    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, ids)]
                    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
                finally:
                    if os.path.exists(tmp_name): os.remove(tmp_name)
            
            return "Error: Unknown model"
        except Exception as e:
            print(f"❌ Error GPU {gpu_idx}: {e}")
            return f"Error: {str(e)}"

manager = ModelManager()
manager.load("moondream2")
print('\n✅ ModelManager ready (v3.1 Final)')

In [ ]:
# ── 4. Streaming Video Processing ─────────────────────────────
import cv2
import numpy as np
from typing import List, Tuple, Generator

def extract_frames(path, interval=1.0, max_w=512):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(int(fps * interval), 1)
    dur = total / fps if total > 0 else 0
    print(f'  Video: {dur:.1f}s, {fps:.0f}fps, every {interval}s')
    frames, idx = [], 0
    while True:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, f = cap.read()
        if not ok: break
        ts = idx / fps
        h, w = f.shape[:2]
        if w > max_w:
            s = max_w / w
            f = cv2.resize(f, (max_w, int(h * s)))
        frames.append((ts, Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))))
        idx += step
    cap.release()
    print(f'  Extracted {len(frames)} frames')
    return frames

def analyze_video_stream(path, prompt, model_name=None, interval=1.0) -> Generator[dict, None, None]:
    if model_name and model_name != manager.current:
        manager.load(model_name)
    
    frames = extract_frames(path, interval)
    n_workers = len(manager.models)
    
    print(f"  Streaming with {n_workers} GPU worker(s)...")
    
    def process_frame(idx, ts, img):
        t0 = time.time()
        gpu_id = idx % n_workers
        desc = manager.query_single(img, prompt, gpu_id)
        dt = time.time() - t0
        mm, ss = int(ts // 60), int(ts % 60)
        tstamp = f'{mm:02d}:{ss:02d}'
        print(f'  Frame {idx+1}/{len(frames)} @ {tstamp} (GPU{gpu_id}) → {desc[:40]}... ({dt:.1f}s)')
        return {
            'timestamp': tstamp, 
            'seconds': round(ts, 2),
            'description': desc,
            'frame_index': idx + 1
        }

    # Submit all tasks
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        futures = {executor.submit(process_frame, i, ts, img): i for i, (ts, img) in enumerate(frames)}
        for f in as_completed(futures):
            yield f.result()

print('✅ Streaming Video utilities ready')

---
## Option A: Analyze in Notebook

In [ ]:
# ── 5A. Notebook Analysis ────────────────────────────────────
import os, glob
vids = [f for f in glob.glob('/kaggle/input/**/*.*', recursive=True)
        if os.path.splitext(f)[1].lower() in {'.mp4','.webm','.avi','.mov','.mkv'}]
if vids:
    for i,v in enumerate(vids): print(f'  [{i}] {v} ({os.path.getsize(v)/1e6:.1f}MB)')
    # ── Settings ──
    VIDEO_INDEX = 0
    PROMPT = 'Describe this scene.'
    MODEL = 'moondream2'
    # ──────────────
    print(f"\n🔍 Analyzing with {MODEL}...")
    for res in analyze_video_stream(vids[VIDEO_INDEX], PROMPT, MODEL):
        pass # Just printing to console
else:
    print('No videos found. Upload via +Add Data.')

---
## Option B: Web UI

In [ ]:
# ── 5B-1. ngrok ──────────────────────────────────────────────
NGROK_AUTH_TOKEN = ''
if NGROK_AUTH_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [ ]:
# ── 5B-2. UI Template (Streaming + History) ──────────────────
FULL_HTML = r"""
<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1.0">
<title>VideoAI Streaming</title>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap" rel="stylesheet">
<style>
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
:root{--bg:#0a0a0f;--card:rgba(18,18,28,.75);--card-h:rgba(25,25,40,.85);--border:rgba(255,255,255,.06);--border-h:rgba(255,255,255,.12);--text:#e8e8f0;--text2:#8888a8;--muted:#555570;--accent:#7c5cfc;--glow:rgba(124,92,252,.3);--green:#5cf0c8;--danger:#f05868;--radius:16px;--sm:10px;--tr:.25s cubic-bezier(.4,0,.2,1);--font:'Inter',sans-serif}
body{font-family:var(--font);background:var(--bg);color:var(--text);min-height:100vh;line-height:1.6}
.glow{position:fixed;border-radius:50%;filter:blur(120px);opacity:.4;pointer-events:none;z-index:0}
.g1{width:500px;height:500px;background:radial-gradient(circle,var(--accent) 0%,transparent 70%);top:-150px;right:-100px;animation:f1 20s ease-in-out infinite}
.g2{width:400px;height:400px;background:radial-gradient(circle,var(--green) 0%,transparent 70%);bottom:-100px;left:-100px;animation:f2 25s ease-in-out infinite}
@keyframes f1{0%,100%{transform:translate(0,0)}50%{transform:translate(-60px,40px)}}
@keyframes f2{0%,100%{transform:translate(0,0)}50%{transform:translate(40px,-60px)}}
header{position:relative;z-index:10;display:flex;align-items:center;justify-content:space-between;padding:20px 32px;border-bottom:1px solid var(--border);backdrop-filter:blur(20px);background:rgba(10,10,15,.6)}
.logo{display:flex;align-items:center;gap:12px} .logo h1{font-size:22px;font-weight:700} .accent{color:var(--accent)}
.badge{display:flex;align-items:center;gap:8px;padding:6px 14px;border-radius:20px;background:var(--card);border:1px solid var(--border);font-size:13px;color:var(--text2)}
.dot{width:8px;height:8px;border-radius:50%;background:var(--muted);transition:var(--tr)} .dot.on{background:var(--green);box-shadow:0 0 8px var(--green)}
.container{position:relative;z-index:10;display:grid;grid-template-columns:1fr 1fr;gap:24px;max-width:1280px;margin:32px auto;padding:0 24px}
@media(max-width:900px){.container{grid-template-columns:1fr}}
.panel{background:var(--card);border:1px solid var(--border);border-radius:var(--radius);padding:28px;backdrop-filter:blur(24px)} .panel:hover{border-color:var(--border-h)}
.title{font-size:16px;font-weight:600;margin-bottom:20px}
.upload{border:2px dashed var(--border);border-radius:var(--sm);padding:40px 24px;text-align:center;cursor:pointer;transition:var(--tr);margin-bottom:20px}
.upload:hover{border-color:var(--accent);background:rgba(124,92,252,.05)}
.upload-icon{font-size:36px;margin-bottom:8px}
.file-info{display:flex;align-items:center;gap:14px;text-align:left} .file-info .icon{font-size:32px} .file-info .name{font-size:14px;font-weight:500;word-break:break-all} .file-info .size{font-size:12px;color:var(--text2)}
.file-remove{width:32px;height:32px;border:1px solid var(--border);border-radius:8px;background:none;color:var(--text2);cursor:pointer;display:flex;align-items:center;justify-content:center} .file-remove:hover{background:rgba(240,88,104,.1);border-color:var(--danger);color:var(--danger)}
select.model-select{width:100%;padding:12px 16px;border:1px solid var(--border);border-radius:var(--sm);background:rgba(255,255,255,.03);color:var(--text);font-family:var(--font);font-size:14px;margin-bottom:20px;transition:var(--tr);appearance:none;background-image:url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='12' height='12' viewBox='0 0 12 12'%3E%3Cpath fill='%238888a8' d='M6 8L1 3h10z'/%3E%3C/svg%3E");background-repeat:no-repeat;background-position:right 16px center;cursor:pointer}
select.model-select:focus{outline:none;border-color:var(--accent);box-shadow:0 0 0 3px var(--glow)}
select.model-select option{background:#1a1a2e;color:var(--text)}
.prompt{width:100%;padding:12px 16px;border:1px solid var(--border);border-radius:var(--sm);background:rgba(255,255,255,.03);color:var(--text);font-family:var(--font);font-size:14px;resize:vertical;transition:var(--tr);margin-bottom:20px}
.prompt:focus{outline:none;border-color:var(--accent);box-shadow:0 0 0 3px var(--glow)} .prompt::placeholder{color:var(--muted)}
label{display:block;font-size:13px;font-weight:500;color:var(--text2);margin-bottom:8px}
.btn{width:100%;padding:14px;border:none;border-radius:var(--sm);background:linear-gradient(135deg,var(--accent),#9c7cfc);color:#fff;font-size:15px;font-weight:600;cursor:pointer;transition:var(--tr);font-family:var(--font)}
.btn:hover:not(:disabled){transform:translateY(-2px);box-shadow:0 8px 30px var(--glow)} .btn:disabled{opacity:.4;cursor:not-allowed}
.error{margin-top:12px;padding:10px 16px;border-radius:var(--sm);background:rgba(240,88,104,.1);border:1px solid rgba(240,88,104,.2);color:var(--danger);font-size:13px;display:none}
.empty{text-align:center;padding:60px 20px;color:var(--muted)} .empty .icon{font-size:48px;opacity:.5;margin-bottom:16px}
.loading{text-align:center;padding:60px 20px;display:none}
.dots{display:flex;justify-content:center;gap:6px;margin-bottom:20px}
.dots span{width:14px;height:14px;border-radius:50%;background:var(--accent);animation:bounce 1.4s ease-in-out infinite}
.dots span:nth-child(2){animation-delay:.16s} .dots span:nth-child(3){animation-delay:.32s}
@keyframes bounce{0%,80%,100%{transform:scale(.6);opacity:.4}40%{transform:scale(1);opacity:1}}
.results-list{display:flex;flex-direction:column;gap:10px;max-height:520px;overflow-y:auto}
.result-card{display:flex;gap:14px;padding:14px 16px;border-radius:var(--sm);background:rgba(255,255,255,.02);border:1px solid var(--border);animation:fadeUp .4s ease-out both}
.result-card:hover{background:var(--card-h);border-color:var(--border-h)}
@keyframes fadeUp{from{opacity:0;transform:translateY(12px)}to{opacity:1;transform:translateY(0)}}
.ts{flex-shrink:0;padding:4px 10px;border-radius:6px;background:var(--glow);color:var(--accent);font-size:12px;font-weight:600;height:fit-content}
.desc{font-size:14px;line-height:1.6}
.model-badge{background:linear-gradient(135deg,#76B900,#4CAF50);color:white;padding:4px 12px;border-radius:12px;font-size:11px;font-weight:600}
.swap-notice{margin-top:12px;padding:10px 16px;border-radius:var(--sm);background:rgba(124,92,252,.1);border:1px solid rgba(124,92,252,.2);color:var(--accent);font-size:13px;display:none;text-align:center}
.history{position:relative;z-index:10;max-width:1280px;margin:32px auto 0;padding:0 24px}
.hist-header{display:flex;align-items:center;justify-content:space-between;margin-bottom:16px}
.hist-list{display:grid;grid-template-columns:repeat(auto-fill,minmax(320px,1fr));gap:12px}
.hist-card{padding:16px;background:var(--card);border:1px solid var(--border);border-radius:var(--sm);cursor:pointer;transition:var(--tr)}
.hist-card:hover{border-color:var(--accent);transform:translateY(-2px);box-shadow:0 8px 24px rgba(124,92,252,.1)}
.hist-top{display:flex;justify-content:space-between;margin-bottom:8px}
.hist-name{font-size:14px;font-weight:600;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;max-width:85%}
.hist-prompt{font-size:13px;color:var(--text2);overflow:hidden;text-overflow:ellipsis;white-space:nowrap;margin-bottom:10px}
.hist-meta{display:flex;flex-wrap:wrap;gap:8px;font-size:11px;color:var(--muted)}
.hist-meta span{padding:2px 8px;border-radius:4px;background:rgba(255,255,255,.03);border:1px solid var(--border)}
footer{position:relative;z-index:10;text-align:center;padding:24px;color:var(--muted);font-size:13px;border-top:1px solid var(--border);margin-top:40px} footer strong{color:var(--accent)}
.hidden{display:none!important}
</style></head><body>
<div class="glow g1"></div><div class="glow g2"></div>
<header><div class="logo"><span style="font-size:28px">🎥</span><h1>Video<span class="accent">AI</span></h1></div>
<div style="display:flex;gap:12px;align-items:center"><span class="model-badge" id="modelBadge">MOONDREAM2</span><div class="badge"><span class="dot" id="dot"></span><span id="status">Connecting...</span></div></div></header>
<main class="container">
<section class="panel"><h2 class="title">📤 Input</h2>
<div class="upload" id="dropZone" onclick="document.getElementById('fileInput').click()">
<div id="uploadContent"><div class="upload-icon">☁️</div><p style="font-size:15px;font-weight:500">Drop video or click to upload</p><p style="font-size:13px;color:var(--text2)">MP4, WebM, AVI, MOV — Max 20 MB</p></div>
<div id="fileSelected" class="file-info hidden"></div></div>
<input type="file" id="fileInput" accept=".mp4,.webm,.avi,.mov,.mkv" hidden>
<div><label>🤖 Model</label><select class="model-select" id="modelSelect"><option value="moondream2">Moondream2 (~4GB)</option><option value="smolvlm">SmolVLM (~4GB)</option><option value="qwen25vl">Qwen2.5-VL 3B (~6GB)</option><option value="qwen3vl">Qwen3-VL 2B (~4GB)</option></select></div>
<div id="swapNotice" class="swap-notice">⏳ Model swap may take 30-60s on first use</div>
<div><label>💬 Prompt</label><textarea class="prompt" id="prompt" rows="3" placeholder="Describe what is happening in this video..."></textarea></div>
<button class="btn" id="analyzeBtn" disabled>⚡ Analyze Streaming</button>
<div class="error" id="error"></div></section>
<section class="panel"><h2 class="title">📊 Results</h2>
<div id="empty" class="empty"><div class="icon">🔍</div><p>Upload a video and enter a prompt</p></div>
<div class="loading" id="loading"><div class="dots"><span></span><span></span><span></span></div><p id="loadMsg">Analyzing frames...</p></div>
<div class="results-list" id="resultsList"></div></section>
</main>
<section class="history" id="historySection">
<div class="hist-header"><h2 class="title">📜 History</h2><button class="btn" style="width:auto;padding:8px 16px" onclick="loadHistory()">🔄 Refresh</button></div>
<div class="hist-list" id="histList"><div class="empty"><p>No saved analyses yet</p></div></div></section>
<footer><p>Powered by <strong>Moondream2 · SmolVLM · Qwen2.5-VL · Qwen3-VL</strong></p></footer>
<script>
const fileInput=document.getElementById('fileInput'),prompt=document.getElementById('prompt'),analyzeBtn=document.getElementById('analyzeBtn'),modelSelect=document.getElementById('modelSelect');
const dropZone=document.getElementById('dropZone'),uploadContent=document.getElementById('uploadContent'),fileSelected=document.getElementById('fileSelected');
const errorEl=document.getElementById('error'),emptyEl=document.getElementById('empty'),loadingEl=document.getElementById('loading'),loadMsg=document.getElementById('loadMsg');
const resultsList=document.getElementById('resultsList'),histList=document.getElementById('histList');
const dot=document.getElementById('dot'),statusEl=document.getElementById('status'),modelBadge=document.getElementById('modelBadge'),swapNotice=document.getElementById('swapNotice');
let selectedFile=null,currentLoaded='';
const labels={moondream2:'MOONDREAM2',smolvlm:'SMOLVLM',qwen25vl:'QWEN2.5-VL',qwen3vl:'QWEN3-VL'};
fetch('/api/health').then(r=>r.json()).then(d=>{dot.classList.add('on');statusEl.textContent=`${d.gpu_count} GPU(s)`;currentLoaded=d.current_model||'';}).catch(()=>{statusEl.textContent='Offline';});
modelSelect.addEventListener('change',()=>{const v=modelSelect.value;modelBadge.textContent=labels[v]||v.toUpperCase();swapNotice.style.display=(v!==currentLoaded)?'block':'none';});
function setFile(f){if(!f)return;selectedFile=f;uploadContent.classList.add('hidden');fileSelected.classList.remove('hidden');fileSelected.innerHTML=`<span class="icon">🎬</span><div style="flex:1"><div class="name">${esc(f.name)}</div><div class="size">${(f.size/1e6).toFixed(1)} MB</div></div><button class="file-remove" onclick="event.stopPropagation();clearFile()">✕</button>`;updateBtn();}
function clearFile(){selectedFile=null;fileInput.value='';uploadContent.classList.remove('hidden');fileSelected.classList.add('hidden');updateBtn();}
function updateBtn(){analyzeBtn.disabled=!(selectedFile&&prompt.value.trim());}
function esc(s){const d=document.createElement('div');d.textContent=s;return d.innerHTML;}
fileInput.addEventListener('change',e=>{if(e.target.files[0])setFile(e.target.files[0]);});
prompt.addEventListener('input',updateBtn);
dropZone.addEventListener('dragover',e=>{e.preventDefault();dropZone.style.borderColor='var(--accent)';});
dropZone.addEventListener('dragleave',()=>{dropZone.style.borderColor='';});
dropZone.addEventListener('drop',e=>{e.preventDefault();dropZone.style.borderColor='';if(e.dataTransfer.files[0])setFile(e.dataTransfer.files[0]);});
analyzeBtn.addEventListener('click',async()=>{
  errorEl.style.display='none';emptyEl.classList.add('hidden');resultsList.innerHTML='';analyzeBtn.disabled=true;
  loadingEl.style.display='block';loadMsg.textContent='Initializing stream...';swapNotice.style.display='none';
  const fd=new FormData();fd.append('video',selectedFile);fd.append('prompt',prompt.value);fd.append('model',modelSelect.value);
  try{
    const response = await fetch('/api/analyze-stream',{method:'POST',body:fd});
    if(!response.ok) throw new Error('Stream failed');
    loadingEl.style.display='none';
    const reader = response.body.getReader();
    const decoder = new TextDecoder();
    while(true){
        const {value,done} = await reader.read();
        if(done) break;
        const chunk = decoder.decode(value);
        const lines = chunk.split('\n');
        for(const line of lines){
            if(line.startsWith('data: ')){
                try{
                    const data = JSON.parse(line.substring(6));
                    addResultCard(data);
                }catch(e){}
            }
        }
    }
    loadHistory();
  } catch(e){errorEl.textContent=e.message;errorEl.style.display='block';loadingEl.style.display='none';emptyEl.classList.remove('hidden');}
  finally{analyzeBtn.disabled=false;updateBtn();currentLoaded=modelSelect.value;}
});
function addResultCard(r){
    if(!r.timestamp) return;
    const div = document.createElement('div'); div.className='result-card';
    div.innerHTML=`<span class="ts">${r.timestamp}</span><span class="desc">${esc(r.description)}</span>`;
    resultsList.appendChild(div); 
    // Auto-scroll to bottom
    const panel = resultsList.parentElement;
    panel.scrollTop = panel.scrollHeight;
}
async function loadHistory(){try{const r=await fetch('/api/history');const d=await r.json();const a=d.analyses||[];if(!a.length){histList.innerHTML='<div class="empty"><p>No saved analyses yet</p></div>';return;}histList.innerHTML=a.map(h=>`<div class="hist-card" onclick="loadResult(${JSON.stringify(h.results).replace(/"/g,'&quot;')})"><div class="hist-top"><span class="hist-name">🎬 ${esc(h.video_filename||'')}</span></div><div class="hist-prompt">${esc(h.prompt||'')}</div><div class="hist-meta"><span>${h.frames_analyzed||0} frames</span><span>${h.model||'unk'}</span></div></div>`).join('');}catch(e){console.error(e);}}
function loadResult(results){
  resultsList.innerHTML='';
  if(Array.isArray(results)) results.forEach(addResultCard);
  window.scrollTo({top:0,behavior:'smooth'});
}
loadHistory();
</script></body></html>
"""
print('✅ Streaming UI template loaded')

In [ ]:
# ── 5B-3. Server + Streaming Endpoint ────────────────────────
import os, uuid, time, threading, subprocess, json
from datetime import datetime
from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.responses import HTMLResponse, StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

# Kill old server
try: subprocess.run(['fuser','-k','8000/tcp'],capture_output=True); time.sleep(1)
except: pass
try:
    from pyngrok import ngrok as _ng
    for t in _ng.get_tunnels(): _ng.disconnect(t.public_url)
except: pass

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

HISTORY = []

@app.get('/api/health')
async def health():
    return {'status':'ok','device':'cuda','gpu_count':len(manager.models),'current_model':manager.current}

async def stream_generator(path, prompt, model):
    final_results = []
    # Yield frames as they complete
    for res in analyze_video_stream(path, prompt, model_name=model):
        final_results.append(res)
        # Send SSE event per frame
        yield f"data: {json.dumps(res)}\n\n"
    
    # Save to history when done
    HISTORY.insert(0, {
        'video_filename': os.path.basename(path),
        'prompt': prompt, 
        'model': model,
        'frames_analyzed': len(final_results),
        'timestamp': datetime.now().isoformat(),
        'results': sorted(final_results, key=lambda x: x['frame_index'])
    })

@app.post('/api/analyze-stream')
async def api_analyze_stream(video:UploadFile=File(...), prompt:str=Form(...), model:str=Form('moondream2')):
    data = await video.read()
    if len(data)>20*1024*1024: raise HTTPException(413,'Max 20MB')
    ext = os.path.splitext(video.filename or '.mp4')[1]
    path = f'/tmp/{uuid.uuid4().hex}{ext}'
    
    # Save file
    with open(path,'wb') as f: f.write(data)
    
    # Return streaming response
    return StreamingResponse(stream_generator(path, prompt, model), media_type="text/event-stream")

@app.get('/api/history')
async def get_history():
    return {'analyses': HISTORY}

@app.get('/',response_class=HTMLResponse)
async def serve_ui(): return FULL_HTML

threading.Thread(target=lambda:uvicorn.run(app,host='0.0.0.0',port=8000),daemon=True).start()
time.sleep(3)

if NGROK_AUTH_TOKEN:
    t = ngrok.connect(8000)
    print(f'\n🌐 UI: {t.public_url}\n')
else:
    print('⚠️ No ngrok token')